# 035 — HiPace_QMS (HiPace450) Comms Test

Quick check after the RS485-USB adapter swap (2026-10-23). Read-only: connect, a few hk reads, disconnect. No start/stop/vent.

**Before running:** the `main_turbos` service owns COM27 — the dashboard *Telemetry* toggle does NOT free the port.
Stop the service first (dashboard System card -> `main_turbos` -> Stop, or `POST http://127.0.0.1:8810/process/main_turbos/stop`).
This also pauses Transfer/Depo telemetry; the pumps themselves keep running. Restart the service when done (last cell).

## 1. Imports

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'src'))

import serial
from serial.tools import list_ports
from devices.pfeiffer.hipacebus import HiPace300Bus   # HiPace450 = same TC400 frame
from devices.pfeiffer import pfeifferVacuumProtocol as pvp
pvp.enable_valid_char_filter()

## 2. Config (values from `lab_services/lab_config.toml`)

In [ ]:
COM_PORT = 'COM27'          # verify in Device Manager — new adapter may have a new number
OMNICONTROL_ADDRESS = 101
TC400_ADDRESS = 1
GAUGE1_ADDRESS = 122        # Collision_Cell gauge
TOLERANT_COMM = True       # CR-tolerant + retrying exchange (TC400 drops ~10-25 % of CRs);
                           # False = stock protocol, use it to reproduce the fault

## 3. Port check
Lists all COM ports, then tries a bare open. `PermissionError ... Access is denied` = port still owned (service running).

In [ ]:
for p in list_ports.comports():
    print(f"{p.device:7s} {p.description}  [{p.hwid}]")

try:
    s = serial.Serial(COM_PORT, 9600, timeout=1); s.close()
    print(f"{COM_PORT}: free")
except Exception as e:
    print(f"{COM_PORT}: {e!r}")

## 4. Connect

In [ ]:
pump = HiPace300Bus(
    device_id='HiPace_QMS_test',
    port=COM_PORT,
    device_address=OMNICONTROL_ADDRESS,
    tc400_address=TC400_ADDRESS,
    gauge1_address=GAUGE1_ADDRESS,
    timeout=1.0,
    tolerant_comm=TOLERANT_COMM,
)
print("connect() ->", pump.connect())

## 5. Housekeeping reads

In [ ]:
print("OmniControl name   :", pump.get_omni_device_name())
print("OmniControl RS485  :", pump.get_omni_rs485_address())
print("OmniControl error  :", pump.get_omni_error_code())
print("TC400 name         :", pump.get_pump_device_name())
print("TC400 error        :", pump.get_pump_error_code())

In [ ]:
print("Pump station on    :", pump.get_pumpStatn_enabled())
print("Motor pump on      :", pump.get_motor_pump_enabled())
#print("Actual speed       :", pump.get_actual_speed_hz(), "Hz")
print("Target reached     :", pump.is_target_speed_reached())
print("Drive power        :", pump.get_drive_power(), "W")
print("Temp electronics   :", pump.get_electronics_temperature(), "degC")
print("Temp pump bottom   :", pump.get_pump_bottom_temperature(), "degC")
print("Vent enabled       :", pump.get_vent_enabled())

In [ ]:
print("Gauge sensor on    :", pump.get_SensOnOff())
print("Collision_Cell     :", pump.get_gauge_pressure(), "hPa")

In [ ]:
pump.get_status()

## 5b. Bus diagnostic — raw frame capture

Sends the same read-only query N times per address and sorts every reply:
- `clean` = full frame ending in CR
- `empty` = no reply at all (request lost on the wire)
- `no_CR` = checksum-valid frame, but the final CR is missing/corrupted (e.g. `0xC3`)
- `garbage` = anything else

Mostly `empty` + `no_CR` = the unbiased-RS-485 signature from the HiPace80 bus (KB pfeiffer, 2026-07-09).
Errors on one address only = that device/node, not the adapter.

In [ ]:
import time
from collections import Counter

N = 30
TARGETS = {                      # read-only queries: [P:349] device name
    'omnicontrol': (OMNICONTROL_ADDRESS, 349),
    'tc400':       (TC400_ADDRESS, 349),
    'gauge1':      (GAUGE1_ADDRESS, 349),
}

def raw_query(s, addr, param):
    c = "{:03d}00{:03d}02=?".format(addr, param)
    c += "{:03d}\r".format(sum(map(ord, c)) % 256)
    s.reset_input_buffer()
    s.write(c.encode())
    return s.read(64)            # returns after 64 bytes or s.timeout

def classify(b):
    if not b:
        return 'empty'
    if b.endswith(b'\r'):
        return 'clean'
    try:
        r = b[:-1].decode('ascii') if b[-1] > 127 else b.decode('ascii')
    except UnicodeDecodeError:
        return 'garbage'
    if len(r) >= 13 and r[-3:].isdigit() and int(r[-3:]) == sum(map(ord, r[:-3])) % 256:
        return 'no_CR'
    return 'garbage'

s = pump.serial_connection
old_timeout, s.timeout = s.timeout, 0.2
try:
    for name, (addr, param) in TARGETS.items():
        tally, bad = Counter(), []
        for _ in range(N):
            b = raw_query(s, addr, param)
            k = classify(b)
            tally[k] += 1
            if k != 'clean' and len(bad) < 3:
                bad.append(b)
            time.sleep(0.05)
        print(f"{name:12s} addr {addr:3d}: {dict(tally)}")
        for b in bad:
            print(f"    {b!r}  hex={b.hex(' ')}")
finally:
    s.timeout = old_timeout

## 5c. Same queries through the tolerant reader (HiPace80DCU fix)

Accepts checksum-valid frames without CR + flushes and retries (3 attempts). If this gets ~100 % while 5b is flaky,
the software workaround also works for this bus. The proper fix is still hardware (bias/termination/GND, see notes).

In [ ]:
from devices.pfeiffer.hipacebus.tolerant_comm import exchange_with_retry

for name, (addr, param) in TARGETS.items():
    ok, errs = 0, Counter()
    for _ in range(N):
        try:
            exchange_with_retry(pump.serial_connection, addr, param, attempts=3, retry_delay=0.05)
            ok += 1
        except ValueError as e:
            errs[str(e).split(':')[-1].strip()] += 1
    print(f"{name:12s} addr {addr:3d}: {ok}/{N} ok  {dict(errs)}")

## 6. Disconnect + hand port back

In [ ]:
pump.disconnect()
# then restart the service: dashboard System card -> main_turbos -> Start
# or: import urllib.request; urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8810/process/main_turbos/start', method='POST')).read()